# Sistema de Vigilância Respiratória - Análise Interativa

**Autor**: Dr. Leandro (HUSF) 
**Data**: Março 2025  
**Objetivo**: Análise epidemiológica interativa para orientação de isolamento respiratório

---

Este notebook permite análise interativa dos dados epidemiológicos de SRAG/SG para calcular pressão epidemiológica e orientar decisões sobre Valor Preditivo Negativo (VPN) de testes de antígeno.

## 1. Configuração Inicial

In [ ]:
# Importações necessárias
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import json
from pathlib import Path

# Configurações
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Adicionar diretório do sistema ao path (ajustar conforme instalação)
sys.path.append('/home/claude')
sys.path.append('/opt/vigilancia_respiratoria/scripts')

print("✓ Importações realizadas")
print(f"✓ Data/hora atual: {datetime.now().strftime('%d/%m/%Y %H:%M')}")

In [ ]:
# Importar módulos do sistema de vigilância
try:
    from sistema_vigilancia_respiratoria import (
        SistemaVigilanciaRespiratoria,
        ConfiguradorEpidemiologico,
        ExtractorDadosEpidemiologicos,
        CalculadorPressaoEpidemiologica,
        OrientadorIsolamento
    )
    
    from analise_tendencias import (
        AnalisadorTendencias,
        SistemaInteligentePreditivo,
        ParametrosModelagem
    )
    
    print("✓ Módulos do sistema carregados com sucesso")
    
except ImportError as e:
    print(f"❌ Erro ao importar módulos: {e}")
    print("Execute primeiro: python setup_vigilancia.py --hospital husf_campinas")

## 2. Configuração do Hospital

In [ ]:
# Configuração para HUSF - Campinas
config = ConfiguradorEpidemiologico(
    codigo_municipio="3543402",  # Campinas-SP
    codigo_estado="35",         # São Paulo
    nome_regiao="Região Metropolitana de Campinas - HUSF",
    sensibilidade_antigeno=0.85,
    especificidade_antigeno=0.98,
    limiar_baixa_circulacao=0.05,
    limiar_media_circulacao=0.15,
    limiar_alta_circulacao=0.25,
    janela_analise_dias=14,
    janela_tendencia_dias=28
)

print("Configuração carregada:")
print(f"• Região: {config.nome_regiao}")
print(f"• Município: {config.codigo_municipio}")
print(f"• Sensibilidade teste antígeno: {config.sensibilidade_antigeno:.0%}")
print(f"• Especificidade teste antígeno: {config.especificidade_antigeno:.0%}")
print(f"• Janela de análise: {config.janela_analise_dias} dias")

## 3. Extração e Análise de Dados

In [ ]:
# Inicializar sistema de vigilância
sistema = SistemaVigilanciaRespiratoria(config)

print("Sistema de vigilância inicializado")
print("Iniciando extração de dados...")

# Executar análise completa
resultados = sistema.executar_analise_completa()

if resultados:
    print("✓ Análise concluída com sucesso")
    print(f"• Casos analisados: {resultados['total_casos_analisados']}")
    print(f"• Período: {resultados['periodo_dados']}")
    print(f"• Região: {resultados['regiao']}")
else:
    print("❌ Falha na análise - verificar conectividade e dados")

In [ ]:
# Visualizar positividade por patógeno
if resultados and 'positividade' in resultados:
    positividade = resultados['positividade']
    
    plt.figure(figsize=(12, 6))
    
    # Gráfico de barras
    patogenos = list(positividade.keys())
    valores = [positividade[p] * 100 for p in patogenos]
    cores = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
    
    bars = plt.bar(patogenos, valores, color=cores[:len(patogenos)])
    
    # Adicionar valores nas barras
    for bar, val in zip(bars, valores):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    # Linhas de referência
    plt.axhline(y=config.limiar_baixa_circulacao*100, color='green', 
                linestyle='--', alpha=0.7, label='Limiar baixa circulação (5%)')
    plt.axhline(y=config.limiar_media_circulacao*100, color='orange', 
                linestyle='--', alpha=0.7, label='Limiar média circulação (15%)')
    plt.axhline(y=config.limiar_alta_circulacao*100, color='red', 
                linestyle='--', alpha=0.7, label='Limiar alta circulação (25%)')
    
    plt.title('Positividade por Patógeno - HUSF', fontsize=16, fontweight='bold')
    plt.ylabel('Positividade (%)', fontweight='bold')
    plt.xlabel('Patógeno', fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Tabela de positividade
    df_positividade = pd.DataFrame([
        {'Patógeno': pat.upper(), 'Positividade (%)': f"{pos*100:.1f}"}
        for pat, pos in positividade.items()
    ])
    
    print("\nTabela de Positividade:")
    print(df_positividade.to_string(index=False))

## 4. Análise de Valor Preditivo Negativo (VPN)

In [ ]:
# Visualizar VPN por patógeno
if resultados and 'vpn_valores' in resultados:
    vpn_data = resultados['vpn_valores']
    
    plt.figure(figsize=(12, 6))
    
    patogenos = list(vpn_data.keys())
    vpn_valores = [vpn_data[p]['vpn'] * 100 for p in patogenos]
    cores = ['#FFB347', '#98D8E8', '#F7DC6F', '#BB8FCE']
    
    bars = plt.bar(patogenos, vpn_valores, color=cores[:len(patogenos)])
    
    # Adicionar valores nas barras
    for bar, val in zip(bars, vpn_valores):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    # Linhas de referência para VPN
    plt.axhline(y=90, color='orange', linestyle='--', alpha=0.7, label='VPN 90% (Mínimo)')
    plt.axhline(y=95, color='green', linestyle='--', alpha=0.7, label='VPN 95% (Ideal)')
    
    plt.title('Valor Preditivo Negativo (VPN) - Teste de Antígeno', fontsize=16, fontweight='bold')
    plt.ylabel('VPN (%)', fontweight='bold')
    plt.xlabel('Patógeno', fontweight='bold')
    plt.ylim(80, 100)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Tabela de VPN
    df_vpn = pd.DataFrame([
        {
            'Patógeno': pat.upper(), 
            'VPN (%)': f"{vpn_data[pat]['vpn']*100:.1f}",
            'VPP (%)': f"{vpn_data[pat]['vpp']*100:.1f}",
            'Prevalência (%)': f"{vpn_data[pat]['prevalencia_usada']*100:.1f}"
        }
        for pat in vpn_data.keys()
    ])
    
    print("\nTabela de Valores Preditivos:")
    print(df_vpn.to_string(index=False))

## 5. Pressão Epidemiológica e Orientações

In [ ]:
# Visualizar pressão epidemiológica
if resultados and 'pressao_epidemiologica' in resultados:
    pressao = resultados['pressao_epidemiologica']
    orientacoes = resultados['orientacoes']
    
    # Mapeamento de cores por pressão
    cores_pressao = {
        'BAIXA': '#28a745',      # Verde
        'MODERADA': '#ffc107',   # Amarelo
        'ALTA': '#fd7e14',       # Laranja
        'MUITO ALTA': '#dc3545'  # Vermelho
    }
    
    plt.figure(figsize=(14, 8))
    
    # Subplot 1: Pressão epidemiológica
    plt.subplot(2, 1, 1)
    
    patogenos = list(pressao.keys())
    pressao_num = []
    cores_bars = []
    
    mapa_pressao = {'BAIXA': 1, 'MODERADA': 2, 'ALTA': 3, 'MUITO ALTA': 4}
    
    for pat in patogenos:
        nivel = pressao[pat]
        pressao_num.append(mapa_pressao[nivel])
        cores_bars.append(cores_pressao[nivel])
    
    bars = plt.bar(patogenos, pressao_num, color=cores_bars)
    
    # Adicionar rótulos de pressão
    for i, (bar, pat) in enumerate(zip(bars, patogenos)):
        plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                pressao[pat], ha='center', va='bottom', fontweight='bold')
    
    plt.title('Pressão Epidemiológica por Patógeno', fontsize=14, fontweight='bold')
    plt.ylabel('Nível de Pressão')
    plt.ylim(0, 5)
    plt.yticks([1, 2, 3, 4], ['BAIXA', 'MODERADA', 'ALTA', 'MUITO ALTA'])
    
    # Subplot 2: Orientações (texto)
    plt.subplot(2, 1, 2)
    plt.axis('off')
    
    orientacoes_text = "\n".join([
        f"• {pat.upper()}: {orient}"
        for pat, orient in orientacoes.items()
    ])
    
    plt.text(0, 1, "ORIENTAÇÕES DE ISOLAMENTO:\n\n" + orientacoes_text, 
             transform=plt.gca().transAxes, fontsize=11,
             verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
    
    plt.tight_layout()
    plt.show()
    
    # Tabela de orientações
    df_orientacoes = pd.DataFrame([
        {
            'Patógeno': pat.upper(),
            'Pressão Epidemiológica': pressao[pat],
            'Orientação': orient
        }
        for pat, orient in orientacoes.items()
    ])
    
    print("\nTabela de Orientações:")
    print(df_orientacoes.to_string(index=False, max_colwidth=60))

## 6. Análise de Tendências (se dados históricos disponíveis)

In [ ]:
# Tentar análise de tendências
try:
    sistema_preditivo = SistemaInteligentePreditivo()
    resultado_tendencias = sistema_preditivo.executar_analise_preditiva_completa()
    
    if 'erro' not in resultado_tendencias:
        print("✓ Análise de tendências concluída")
        print(f"• Período analisado: {resultado_tendencias['periodo_analise']}")
        print(f"• Alertas gerados: {len(resultado_tendencias['alertas'])}")
        
        # Mostrar alertas se houver
        if resultado_tendencias['alertas']:
            print("\n🚨 ALERTAS AUTOMÁTICOS:")
            for alerta in resultado_tendencias['alertas']:
                severidade_emoji = {'ALTA': '🔴', 'MEDIA': '🟡', 'BAIXA': '🟢'}
                emoji = severidade_emoji.get(alerta['severidade'], '⚪')
                print(f"{emoji} {alerta['tipo']} - {alerta['patogeno'].upper()}: {alerta['mensagem']}")
        
        # Exibir gráfico de tendências se gerado
        import os
        if os.path.exists('/home/claude/tendencias_epidemiologicas.png'):
            from IPython.display import Image, display
            display(Image('/home/claude/tendencias_epidemiologicas.png'))
    
    else:
        print(f"⚠️ {resultado_tendencias['erro']}")
        print(f"Recomendação: {resultado_tendencias['recomendacao']}")
        
except Exception as e:
    print(f"⚠️ Análise de tendências não disponível: {str(e)}")
    print("Execute o sistema por algumas semanas para acumular dados históricos.")

## 7. Simulador de Cenários

In [ ]:
# Simulador interativo de VPN para diferentes cenários de prevalência
def simular_vpn_cenarios(sensibilidade=0.85, especificidade=0.98):
    """Simula VPN para diferentes cenários de prevalência"""
    
    prevalencias = np.arange(0.01, 0.50, 0.01)  # 1% a 50%
    vpn_valores = []
    vpp_valores = []
    
    for prev in prevalencias:
        # VPN = (espec × (1-prev)) / (espec × (1-prev) + (1-sens) × prev)
        vpn = (especificidade * (1 - prev)) / (especificidade * (1 - prev) + (1 - sensibilidade) * prev)
        
        # VPP = (sens × prev) / (sens × prev + (1-espec) × (1-prev))
        vpp = (sensibilidade * prev) / (sensibilidade * prev + (1 - especificidade) * (1 - prev))
        
        vpn_valores.append(vpn)
        vpp_valores.append(vpp)
    
    return prevalencias, vpn_valores, vpp_valores

# Executar simulação
prev, vpn, vpp = simular_vpn_cenarios(config.sensibilidade_antigeno, config.especificidade_antigeno)

# Plotar resultados
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(prev * 100, np.array(vpn) * 100, 'b-', linewidth=2, label='VPN')
plt.axhline(y=90, color='orange', linestyle='--', alpha=0.7, label='VPN 90%')
plt.axhline(y=95, color='green', linestyle='--', alpha=0.7, label='VPN 95%')

# Marcar prevalências atuais se disponível
if resultados and 'positividade' in resultados:
    for patogeno, pos in resultados['positividade'].items():
        vpn_atual = config.especificidade_antigeno * (1 - pos) / (
            config.especificidade_antigeno * (1 - pos) + (1 - config.sensibilidade_antigeno) * pos
        )
        plt.scatter(pos * 100, vpn_atual * 100, s=100, label=f'{patogeno.upper()} atual')

plt.title('VPN vs Prevalência - Teste de Antígeno', fontweight='bold')
plt.xlabel('Prevalência (%)')
plt.ylabel('VPN (%)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(prev * 100, np.array(vpp) * 100, 'r-', linewidth=2, label='VPP')

# Marcar VPP atuais se disponível
if resultados and 'vpn_valores' in resultados:
    for patogeno, vpn_data in resultados['vpn_valores'].items():
        if 'vpp' in vpn_data:
            prevalencia_atual = vpn_data['prevalencia_usada']
            vpp_atual = vpn_data['vpp']
            plt.scatter(prevalencia_atual * 100, vpp_atual * 100, s=100, label=f'{patogeno.upper()} atual')

plt.title('VPP vs Prevalência - Teste de Antígeno', fontweight='bold')
plt.xlabel('Prevalência (%)')
plt.ylabel('VPP (%)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSimulação baseada em:")
print(f"• Sensibilidade: {config.sensibilidade_antigeno:.0%}")
print(f"• Especificidade: {config.especificidade_antigeno:.0%}")
print(f"\n• VPN > 95%: Prevalência < {prev[np.array(vpn) > 0.95][-1]*100:.1f}%")
print(f"• VPN > 90%: Prevalência < {prev[np.array(vpn) > 0.90][-1]*100:.1f}%")

## 8. Relatório Final

In [ ]:
# Gerar e exibir relatório final
if resultados:
    print("═" * 80)
    print("RELATÓRIO FINAL - VIGILÂNCIA RESPIRATÓRIA HUSF")
    print("═" * 80)
    
    print(resultados['relatorio'])
    
    # Salvar resultados
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # JSON completo
    with open(f'/home/claude/analise_interativa_{timestamp}.json', 'w') as f:
        json.dump(resultados, f, indent=2, default=str, ensure_ascii=False)
    
    # Relatório markdown
    with open(f'/home/claude/relatorio_interativo_{timestamp}.md', 'w') as f:
        f.write(resultados['relatorio'])
    
    print(f"\n✓ Resultados salvos com timestamp: {timestamp}")
    print(f"✓ Análise concluída em: {datetime.now().strftime('%d/%m/%Y %H:%M')}")

else:
    print("❌ Não foi possível gerar relatório - verificar dados e conectividade")

## 9. Próximos Passos

**Para uso contínuo:**

1. **Automatização**: Configure o cron job para execução quinzenal
   ```bash
   sudo python setup_vigilancia.py --hospital husf_campinas
   ```

2. **Email**: Configure envio automático de relatórios
   ```bash
   python setup_vigilancia.py --configurar-email
   ```

3. **Monitoramento**: Verifique logs regularmente em `/var/log/vigilancia_respiratoria/`

4. **Ajustes**: Modifique limiares conforme necessário no arquivo de configuração

5. **Integração**: Considere integração com sistemas hospitalares existentes

---

**Contato**: Dr. Leandro - CCIH HUSF  
**Email**: leandro.ccih@husf.br  
**Sistema**: Vigilância Respiratória v1.0